# 🎯 02 - Center and Spread

Module 06 used `.describe()` as a first look, to spot problems. Now we read it properly. Every numeric column can be summarized with two questions:

1. **Where is the middle?** (center: mean, median, mode)
2. **How spread out are the values?** (spread: range, standard deviation, IQR, percentiles)

Two columns can have the same center and completely different spreads, and a single number is never the whole story.

## ✅ Learning Goals
- Compute the mean, median, and mode with pandas methods, and explain when they disagree
- Show that the median is **robust** to outliers and the mean is not
- Compute range, variance, standard deviation, IQR, and any percentile
- Explain `ddof`, and why pandas and NumPy give different standard deviations by default
- Read every row of `.describe()`, including what `count` tells you about missing data
- See how filling missing values (Module 06) changes the statistics

In [1]:
import numpy as np
import pandas as pd

titanic = pd.read_csv('titanic.csv')
fare = titanic['Fare']
age = titanic['Age']

## 1. Center: Mean, Median, Mode
Every pandas `Series` has these built in, so no NumPy or SciPy is needed:

| Statistic | Method | Meaning |
|---|---|---|
| Mean | `.mean()` | The sum divided by the count. The "balance point." |
| Median | `.median()` | The middle value after sorting. Half the values are below it. |
| Mode | `.mode()` | The most common value (or values). |

In [2]:
print(f'Fare mean:   {fare.mean():.2f}')
print(f'Fare median: {fare.median():.2f}')
print('Fare mode:  ', fare.mode().tolist())

Fare mean:   32.20
Fare median: 14.45
Fare mode:   [8.05]


The mean fare is more than **twice** the median. That gap is a signal, and the next section shows what causes it.

`.mode()` returns a **Series**, not a single number, because a column can have a tie for most common value. Use `.mode()[0]` when you need just one.

## 2. Robustness: Who Gets Pulled by Outliers?
A statistic is **robust** if a few extreme values barely move it. Watch what happens to a small, made-up set of fares when one very expensive ticket is added:

In [3]:
fares = pd.Series([8, 10, 12, 15, 20])
with_outlier = pd.Series([8, 10, 12, 15, 20, 500])

print('Without outlier: mean', fares.mean(), '| median', fares.median())
print('With outlier:    mean', round(with_outlier.mean(), 1), '| median', with_outlier.median())

Without outlier: mean 13.0 | median 12.0
With outlier:    mean 94.2 | median 13.5


One value moved the mean from 13 to about 94. The median barely changed (12 → 13.5), because it only cares about the **middle position**, not how far away the extreme values are.

That's what is happening in the real Titanic fares: a small number of very expensive 1st-class tickets pull the mean far above what a typical passenger paid.

| Use the… | When… |
|---|---|
| **Median** | the data are lopsided or have extreme values (incomes, prices, house sizes) |
| **Mean** | the data are roughly symmetric, or you need a total (mean × count = total) |
| **Mode** | the data are categories, or a few repeated values (most common shoe size) |

✅ **Your Turn:** Compute the mean and median of `age`. Are they close? What does that suggest about whether age has extreme values?

In [4]:
print(f'Known ages: {age.count()} of {len(age)}; missing: {age.isna().sum()}')
print(f'Age mean: {age.mean():.2f} years')
print(f'Age median: {age.median():.2f} years')
print(f'Mean minus median: {age.mean() - age.median():.2f} years')
print('They are fairly close: the mean is only 1.70 years above the median.')
print('This suggests less upward pull from extreme values than for fares, but does not rule out outliers.')
print(f'Observed age range: {age.min():.2f} to {age.max():.2f} years; inspect the distribution before excluding any age.')


Known ages: 714 of 891; missing: 177
Age mean: 29.70 years
Age median: 28.00 years
Mean minus median: 1.70 years
They are fairly close: the mean is only 1.70 years above the median.
This suggests less upward pull from extreme values than for fares, but does not rule out outliers.
Observed age range: 0.42 to 80.00 years; inspect the distribution before excluding any age.


## 3. Spread: Range, Variance, Standard Deviation
- **Range** = max − min. Easy, but it depends on only the two most extreme values.
- **Variance** = the average squared distance from the mean. Its units are squared (pounds²), which is hard to interpret.
- **Standard deviation (std)** = the square root of the variance, back in the original units. Roughly: how far a typical value is from the mean.

In [5]:
print(f'Age range: {age.max() - age.min():.2f} years ({age.min()} to {age.max()})')
print(f'Age variance: {age.var():.2f} (years squared)')
print(f'Age std: {age.std():.2f} years')

Age range: 79.58 years (0.42 to 80.0)
Age variance: 211.02 (years squared)
Age std: 14.53 years


### `ddof`: Why Pandas and NumPy Disagree
Compute the standard deviation of the same column two ways:

In [6]:
known_ages = age.dropna()

print('pandas .std():', known_ages.std())
print('np.std():     ', np.std(known_ages))

pandas .std(): 14.526497332334042
np.std():      14.516321150817316


They're different, and neither is a bug. The difference is the **`ddof`** ("delta degrees of freedom") setting:

| | Default | Divides by | Use when |
|---|---|---|---|
| pandas `.std()` / `.var()` | `ddof=1` | n − 1 | your data are a **sample** of a bigger population |
| NumPy `np.std()` / `np.var()` | `ddof=0` | n | your data **are** the whole population |

Dividing by n − 1 corrects for the fact that a sample tends to look a little less spread out than the population it came from. Titanic's 891 rows are a sample of the roughly 2,200 people aboard, so pandas' default is the right one here. With large n the difference is tiny. With 5 values, it isn't.

You can always set it explicitly, and the two will match:

In [7]:
print(known_ages.std(ddof=0))
print(np.std(known_ages, ddof=0))

14.516321150817316
14.516321150817316


## 4. Percentiles, Quartiles, and the IQR
A **percentile** answers "what value is this percentage of the data below?" The 90th percentile of fare is the price that 90% of passengers paid less than.

The **quartiles** are the 25th, 50th, and 75th percentiles (Q1, the median, and Q3). The **interquartile range (IQR)** is Q3 − Q1: the width of the **middle half** of the data. Like the median, it ignores the extremes, so it's robust.

In [8]:
fare.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.99])

0.10      7.55000
0.25      7.91040
0.50     14.45420
0.75     31.00000
0.90     77.95830
0.99    249.00622
Name: Fare, dtype: float64

In [9]:
q1 = fare.quantile(0.25)
q3 = fare.quantile(0.75)
iqr = q3 - q1

print(f'Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}')
print(f'Fare std = {fare.std():.2f}')

Q1 = 7.91, Q3 = 31.00, IQR = 23.09
Fare std = 49.69


The middle half of passengers paid within a band about 23 wide, but the standard deviation is about 50. The std is inflated by the same expensive tickets that pulled the mean up. **Robust center (median) goes with robust spread (IQR); mean goes with std.**

Notice the jump from the 90th percentile (about 78) to the 99th (about 250). The top 1% of fares are in a different world from everyone else. Notebook 03 takes this apart.

## 5. Reading `.describe()` Row by Row
You've run `.describe()` since Module 06. Now you know what every row means:

In [10]:
titanic[['Age', 'Fare', 'SibSp']].describe()

,Age,Fare,SibSp
count,714.000000,891.000000,891.000000
mean,29.699118,32.204208,0.523008
std,14.526497,49.693429,1.102743
min,0.420000,0.000000,0.000000
25%,20.125000,7.910400,0.000000
50%,28.000000,14.454200,0.000000
75%,38.000000,31.000000,1.000000
max,80.000000,512.329200,8.000000


| Row | Meaning | What to look for |
|---|---|---|
| `count` | Non-missing values | Lower than the number of rows? Missing data. `Age` has 714 of 891. |
| `mean` / `50%` | Mean and median | Far apart? Lopsided data or outliers (`Fare`). |
| `std` | Standard deviation | Large relative to the mean? Very spread out. |
| `min` / `max` | The extremes | Impossible values? (A negative age, a $0 fare.) |
| `25%` / `75%` | Q1 and Q3 | Their difference is the IQR. |

You can ask for different percentiles, and turn the table sideways (`.T`) when there are many columns:

In [11]:
titanic[['Age', 'Fare']].describe(percentiles=[0.05, 0.5, 0.95]).T.round(2)

,count,mean,std,min,5%,50%,95%,max
Age,714.0,29.7,14.53,0.42,4.00,28.00,56.00,80.00
Fare,891.0,32.2,49.69,0.00,7.22,14.45,112.08,512.33


### 🧠 Quick Check
`Fare` has a `min` of `0.00`. Is a $0 fare a data error, a placeholder (Module 06), or something real? How could you investigate? (Hint: filter for those rows and look at the other columns.)

**Quick Check answer:** Zero alone does not establish a data error or a missing-value code. Practice 3 below filters the 15 zero-fare records, examines class/age/ticket/embarkation, compares statistics with and without them, and explains the evidence needed before changing them.


## 6. Summarizing Categorical Columns
`.describe()` on text columns gives a different summary: count, number of distinct values, the most common value (`top`), and how often it appears (`freq`).

In [12]:
titanic[['Sex', 'Embarked']].describe()

,Sex,Embarked
count,891,889
unique,2,3
top,male,S
freq,577,644


In [13]:
# The proportion in each category: the categorical version of a "summary"
titanic['Embarked'].value_counts(normalize=True, dropna=False).round(3)

Embarked
S      0.723
C      0.189
Q      0.086
NaN    0.002
Name: proportion, dtype: float64

## 7. Missing Values Change the Statistics
Pandas statistics **skip** missing values by default (`skipna=True`). So `age.mean()` is the mean of the **714 known ages**, not all 891 passengers.

In Module 06 you learned to fill missing values. Watch what a "harmless" median fill does to the spread:

In [14]:
age_filled = age.fillna(age.median())

comparison = pd.DataFrame({
    'original': age.describe(),
    'median-filled': age_filled.describe(),
}).round(2)
comparison

,original,median-filled
count,714.00,891.00
mean,29.70,29.36
std,14.53,13.02
min,0.42,0.42
25%,20.12,22.00
50%,28.00,28.00
75%,38.00,35.00
max,80.00,80.00


The median didn't move, but the **std shrank** and the quartiles squeezed toward the middle. You added 177 identical values right at the center, so the data look *less* spread out than they really are.

> ⚠️ Filling missing values is a decision with consequences. If you fill, say so, and compare the statistics before and after, the way the Module 06 lab compared raw and cleaned data.

---
### 📝 Summary
- Center: `.mean()`, `.median()`, `.mode()`. The median is robust to outliers, and the mean is not.
- Spread: range, `.var()`, `.std()`, `.quantile()`, and IQR = Q3 − Q1. Pair median with IQR, and mean with std.
- Pandas uses `ddof=1` (sample) and NumPy uses `ddof=0` (population). Set it explicitly when it matters.
- `.describe()`: `count` reveals missing data, and a big mean-vs-median gap reveals lopsided data.
- Statistics skip `NaN` by default, and filling missing values can quietly shrink the spread.

## 🏋️ Practice

### Practice 1 — Center of Pokémon Stats (easy)
Load `pokemon.csv` into `poke`. For `HP`, `Attack`, and `Speed`, print the mean and median. Which column has the biggest gap between them?

In [15]:
poke = pd.read_csv('pokemon.csv')
center_summary = poke[['HP', 'Attack', 'Speed']].agg(['count', 'mean', 'median']).T
center_summary['absolute_gap'] = (center_summary['mean'] - center_summary['median']).abs()
display(center_summary.round(5))
print('Largest absolute mean-median gap:', center_summary['absolute_gap'].idxmax())
print('HP has the largest gap: 69.25875 - 65 = 4.25875 points.')


,count,mean,median,absolute_gap
HP,800.0,69.25875,65.0,4.25875
Attack,800.0,79.00125,75.0,4.00125
Speed,800.0,68.27750,65.0,3.27750


Largest absolute mean-median gap: HP
HP has the largest gap: 69.25875 - 65 = 4.25875 points.


### Practice 2 — Same Mean, Different Spread (medium)
Create two Series with **the same mean** (for example, both average to 50) but very different standard deviations. Print the mean, std, and IQR of each. In a comment, describe how a group of people with each set of scores would be different.

In [16]:
scores_close = pd.Series([48, 49, 50, 51, 52])
scores_spread = pd.Series([0, 25, 50, 75, 100])
score_summary = pd.DataFrame({
    name: {'mean': s.mean(), 'sample_std': s.std(ddof=1),
           'IQR': s.quantile(0.75) - s.quantile(0.25)}
    for name, s in {'Close': scores_close, 'Spread': scores_spread}.items()
}).T
display(score_summary.round(4))
assert scores_close.mean() == scores_spread.mean() == 50
# Both hypothetical groups average 50. In Close, every person scores near 50;
# in Spread, performance varies from 0 to 100, with much larger differences
# between people. The mean alone hides those differences.


,mean,sample_std,IQR
Close,50.0,1.5811,2.0
Spread,50.0,39.5285,50.0


### Practice 3 — Zero Fares (harder)
Filter `titanic` for rows where `Fare == 0`. How many are there? Look at their `Pclass`, `Age`, `Ticket`, and `Embarked` columns. Then compute the mean and median `Fare` with and without them. In a Markdown cell, say whether you'd treat them as placeholders, and why.

In [17]:
zero_fares = titanic.loc[titanic['Fare'] == 0]
print('Zero-fare records:', len(zero_fares))
display(zero_fares[['Pclass', 'Age', 'Ticket', 'Embarked']])
display(zero_fares['Pclass'].value_counts().sort_index().rename('zero_fare_count'))
print('Missing ages among zero fares:', zero_fares['Age'].isna().sum())
nonzero_fare = fare.loc[fare != 0]
fare_comparison = pd.DataFrame({
    'With zeros': fare.agg(['count', 'mean', 'median']),
    'Without zeros': nonzero_fare.agg(['count', 'mean', 'median']),
}).T
display(fare_comparison.round(5))


Zero-fare records: 15


,Pclass,Age,Ticket,Embarked
179,3,36.0,LINE,S
263,1,40.0,112059,S
271,3,25.0,LINE,S
277,2,NaN,239853,S
302,3,19.0,LINE,S
413,2,NaN,239853,S
466,2,NaN,239853,S
481,2,NaN,239854,S
597,3,49.0,LINE,S
633,1,NaN,112052,S


Pclass
1    5
2    6
3    4
Name: zero_fare_count, dtype: int64

Missing ages among zero fares: 8


,count,mean,median
With zeros,891.0,32.20421,14.4542
Without zeros,876.0,32.75565,14.5000


There are **15 zero-fare records**: 5 in first class, 6 in second, and 4 in third. All embarked at Southampton (`S`); 8 have missing ages. Several share ticket codes, including `LINE` and `239853`, so related ticketing arrangements are possible, but these fields do not establish why the fare is zero.

I would **retain zero fares as recorded and flag them for investigation**, rather than silently recode them as missing placeholders. The dataset alone cannot distinguish genuine free travel, a fare recorded elsewhere, or missing/error codes; check its documentation and original ticket or passenger records. This sensitivity comparison shows the impact: with zeros, mean fare is **32.20421** and median **14.4542** (891 fares); without zeros, mean **32.75565** and median **14.5** (876 fares). No raw records have been altered.


## ✏️ Your Turn
Pick one numeric column from `pokemon.csv`. In 2–3 sentences, describe it using **one center and one spread statistic that fit its shape**, and justify your choice using the gap between its mean and median.

In [18]:
hp = poke['HP']
hp_q1 = hp.quantile(0.25)
hp_q3 = hp.quantile(0.75)
hp_iqr = hp_q3 - hp_q1
print(f'HP: n={hp.count()}, missing={hp.isna().sum()}, mean={hp.mean():.5f}, median={hp.median():.2f}')
print(f'Q1={hp_q1:.2f}, Q3={hp_q3:.2f}, IQR={hp_iqr:.2f}')


HP: n=800, missing=0, mean=69.25875, median=65.00
Q1=50.00, Q3=80.00, IQR=30.00


For the 800 Pokemon records with known HP, the **median is 65 HP** and the **IQR is 30 HP**, with the middle half between 50 and 80 HP. The mean, 69.25875 HP, is 4.25875 above the median, suggesting some upward pull from high HP values; the robust median and IQR describe the typical value and central spread without letting extremes dominate. This gap is a clue about shape rather than proof of the full distribution.


## 🔥 Challenge (Optional) — Median Absolute Deviation
The **median absolute deviation (MAD)** is a robust replacement for the standard deviation: the median of the distances between each value and the median.
```python
mad = (s - s.median()).abs().median()
```
Compute the MAD and the std for `fares` and `with_outlier` from section 2. How much does each one change when the outlier is added?

In [19]:
# Optional challenge: unscaled MAD and sample standard deviation (ddof=1).
robustness_comparison = pd.DataFrame({
    name: {'MAD': (s - s.median()).abs().median(), 'sample_std': s.std(ddof=1)}
    for name, s in {'Without outlier': fares, 'With outlier': with_outlier}.items()
}).T
display(robustness_comparison.round(6))
changes = robustness_comparison.loc['With outlier'] - robustness_comparison.loc['Without outlier']
print('Absolute increases:')
display(changes.round(6))
print('MAD increases from 3 to 4.5 (+1.5); sample std increases from 4.690416 to 198.861174 (+194.170759).')
print('MAD remains stable relative to std; it is not completely unchanged because adding a sixth value shifts the median.')


,MAD,sample_std
Without outlier,3.0,4.690416
With outlier,4.5,198.861174


Absolute increases:


MAD             1.500000
sample_std    194.170759
dtype: float64

MAD increases from 3 to 4.5 (+1.5); sample std increases from 4.690416 to 198.861174 (+194.170759).
MAD remains stable relative to std; it is not completely unchanged because adding a sixth value shifts the median.


### Verification of completed calculations

The following check independently parses the local input files and verifies the completed statistics. Missing ages are excluded for observed-age summaries; the imputation comparison is checked separately.


In [20]:
# Independent verification against the supplied CSV files using Python statistics.
import csv
import statistics
import math

with open('titanic.csv', newline='', encoding='utf-8') as handle:
    titanic_records = list(csv.DictReader(handle))
with open('pokemon.csv', newline='', encoding='utf-8') as handle:
    pokemon_records = list(csv.DictReader(handle))

def numbers(records, column):
    return [float(row[column]) for row in records if row[column] != '']

def linear_quantile(values, fraction):
    ordered = sorted(values)
    position = (len(ordered) - 1) * fraction
    lower = math.floor(position)
    upper = math.ceil(position)
    return ordered[lower] + (ordered[upper] - ordered[lower]) * (position - lower)

def close(actual, expected):
    assert math.isclose(float(actual), expected, rel_tol=1e-12, abs_tol=1e-12), (actual, expected)

ages_reference = numbers(titanic_records, 'Age')
fares_reference = numbers(titanic_records, 'Fare')
assert len(titanic_records) == 891 and len(ages_reference) == 714
assert age.isna().sum() == 177 and fare.isna().sum() == 0
for series, reference in [(age, ages_reference), (fare, fares_reference)]:
    close(series.mean(), statistics.mean(reference))
    close(series.median(), statistics.median(reference))
    close(series.std(ddof=1), statistics.stdev(reference))
    close(series.std(ddof=0), statistics.pstdev(reference))
    close(series.var(ddof=1), statistics.variance(reference))
    close(series.max() - series.min(), max(reference) - min(reference))
    for fraction in [0.10, 0.25, 0.50, 0.75, 0.90, 0.99]:
        close(series.quantile(fraction), linear_quantile(reference, fraction))
assert fare.mode().tolist() == statistics.multimode(fares_reference)
close(known_ages.std(ddof=0), np.std(known_ages.to_numpy(), ddof=0))
zero_reference = [row for row in titanic_records if float(row['Fare']) == 0]
assert len(zero_reference) == len(zero_fares) == 15
assert sum(row['Age'] == '' for row in zero_reference) == 8
positive_reference = [x for x in fares_reference if x != 0]
close(nonzero_fare.mean(), statistics.mean(positive_reference))
close(nonzero_fare.median(), statistics.median(positive_reference))
for column in ['HP', 'Attack', 'Speed']:
    reference = numbers(pokemon_records, column)
    assert len(reference) == poke[column].count() == 800
    close(center_summary.loc[column, 'mean'], statistics.mean(reference))
    close(center_summary.loc[column, 'median'], statistics.median(reference))
hp_reference = numbers(pokemon_records, 'HP')
close(hp_iqr, linear_quantile(hp_reference, 0.75) - linear_quantile(hp_reference, 0.25))
filled_reference = ages_reference + [statistics.median(ages_reference)] * 177
close(age_filled.mean(), statistics.mean(filled_reference))
close(age_filled.median(), statistics.median(filled_reference))
close(age_filled.std(ddof=1), statistics.stdev(filled_reference))
assert age_filled.std(ddof=1) < age.std(ddof=1)
for name, values in [('Close', [48, 49, 50, 51, 52]), ('Spread', [0, 25, 50, 75, 100])]:
    close(score_summary.loc[name, 'mean'], statistics.mean(values))
    close(score_summary.loc[name, 'sample_std'], statistics.stdev(values))
    close(score_summary.loc[name, 'IQR'], linear_quantile(values, 0.75) - linear_quantile(values, 0.25))
for name, values in [('Without outlier', [8, 10, 12, 15, 20]), ('With outlier', [8, 10, 12, 15, 20, 500])]:
    center = statistics.median(values)
    close(robustness_comparison.loc[name, 'MAD'], statistics.median([abs(x - center) for x in values]))
    close(robustness_comparison.loc[name, 'sample_std'], statistics.stdev(values))
pd.testing.assert_frame_equal(titanic, pd.read_csv('titanic.csv'))
pd.testing.assert_frame_equal(poke, pd.read_csv('pokemon.csv'))
print('Independent CSV/statistics verification passed; raw DataFrames are unchanged.')
print(f'Age sample std={statistics.stdev(ages_reference):.8f}; population std={statistics.pstdev(ages_reference):.8f}.')
print(f'Median-filled age sample std={statistics.stdev(filled_reference):.8f}.')


Independent CSV/statistics verification passed; raw DataFrames are unchanged.
Age sample std=14.52649733; population std=14.51632115.
Median-filled age sample std=13.01969655.


---
✅ Next up → [Shape and Outliers](03-Shape_and_Outliers.ipynb): measuring how lopsided a column is, and finding the values that don't fit.